# MSDSS v3／特徵 v5：離線訓練與獨立評估
上傳新版 dataset.zip。主實驗 train/test/validation=450/150/150，另有96筆未見語境 challenge。
一般 test 保留不同模板、共享已見家族；challenge 保留編碼／JS註解邊界家族；Sylius validation 只用新抽樣的外部輸入。Medusa 排除。
預設 MODE="offline"。資料仍未實機驗證；所有結果標示離線分類實驗。未改寫原文或正規化大小寫／空白／編碼。


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, uuid
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("請只上傳一個新版訓練套件 ZIP")
BASE = Path.cwd() / ("msdss-" + uuid.uuid4().hex[:8])
BASE.mkdir()
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for entry in archive.infolist():
        name = PurePosixPath(entry.filename)
        if name.is_absolute() or '..' in name.parts or not (BASE / entry.filename).resolve().is_relative_to(BASE.resolve()):
            raise ValueError("ZIP 含不合法路徑")
    archive.extractall(BASE)
manifest = json.loads((BASE / 'package-manifest.json').read_text())
for name, expected in manifest.items():
    target = (BASE / name).resolve()
    if not target.is_relative_to(BASE.resolve()) or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise ValueError("套件檔案不一致：" + name)
print("資料和程式位於", BASE)
provenance = json.loads((BASE / "data/provenance.json").read_text(encoding="utf-8"))
if provenance.get("evaluation_protocol") != "document-v3-seen-families-heldout-templates":
    raise ValueError("請上傳新版 v3 dataset.zip；舊420筆套件保留作基準。")


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(BASE / 'model/requirements.txt')])

## 檢查資料
離線模式檢查平台、三類標籤、v3 輸入、配對與跨集合洩漏；不要求尚未存在的 HTTP 或 session 證據。


In [ ]:
MODE = "offline"  # 真實 HTTP 驗證改為 "live"
check = subprocess.run([sys.executable, "-X", "utf8", str(BASE / "model/train.py"),
    "--data-dir", str(BASE / "data"), "--evidence-root", str(BASE / "evidence"),
    "--mode", MODE, "--check-only"], capture_output=True, text=True, encoding="utf-8")
READINESS = json.loads(check.stdout) if check.stdout else {}
if check.returncode != 0:
    print(check.stdout or check.stderr)
    raise RuntimeError("資料檢查未通過，請處理 errors；尚未開始訓練。")

print("Data checks passed; mode:", READINESS.get("mode"))
for split, info in READINESS.get("splits", {}).items():
    print(f"{split}: {info['rows']} rows")
for warning in READINESS.get("warnings", []):
    print(warning)


## 訓練
預先固定樹數，不使用 test 或 Sylius validation 選參數。離線成績與平台實機驗證分開解讀。


In [ ]:
from datetime import datetime
MODE = globals().get("MODE", "offline")
TREES = 150
OUTPUT = BASE / "results" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
command = [sys.executable, "-X", "utf8", str(BASE / "model/train.py"),
    "--data-dir", str(BASE / "data"), "--evidence-root", str(BASE / "evidence"),
    "--mode", MODE, "--output", str(OUTPUT), "--trees", str(TREES)]
run = subprocess.run(command, capture_output=True, text=True, encoding="utf-8")
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    print(run.stdout)
    raise RuntimeError("訓練未完成，具體原因已顯示於上方。")

print("Training completed; the next cell shows test and validation results.")


## 三種保留評估分開呈現
一般 test、Sylius validation、未見語境 challenge 不能混成同一分數。共同設定只顯示一次，完整指標與稽核資訊仍保存在下載檔案。


In [ ]:
import json, html
from IPython.display import display, Markdown, HTML
EVALUATION_SPLITS = ['test', 'validation'] + (['challenge'] if (OUTPUT / 'challenge/report.json').is_file() else [])
REPORTS = {split: json.loads((OUTPUT / split / 'report.json').read_text(encoding='utf-8'))
           for split in EVALUATION_SPLITS}
keys = [('accuracy', 'Accuracy'), ('macro_precision', 'Macro Precision'),
        ('macro_recall', 'Macro Recall'), ('macro_f1', 'Macro-F1'),
        ('normal_false_positive_rate', 'Normal false-positive rate'),
        ('attack_detection_recall', 'Attack detection recall'), ('attack_miss_rate', 'Attack miss rate')]
titles = {'test': 'Seen-family test', 'validation': 'Sylius validation', 'challenge': 'Unseen-context challenge'}
lines = ['| Metric | ' + ' | '.join(titles[s] for s in EVALUATION_SPLITS) + ' |', '|---|' + '---:|' * len(EVALUATION_SPLITS)]
for key, title in keys:
    values = [REPORTS[split].get(key) for split in EVALUATION_SPLITS]
    rendered = [f'{value:.4f}' if value is not None else 'N/A' for value in values]
    lines.append('| ' + title + ' | ' + ' | '.join(rendered) + ' |')
display(Markdown('\n'.join(lines)))
lines = ['| Split | Class | Precision | Recall | F1 | N |', '|---|---|---:|---:|---:|---:|']
for split, report in REPORTS.items():
    for label in ('normal', 'sqli', 'xss'):
        r = report['per_class'][label]
        lines.append(f"| {split} | {label} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |")
display(Markdown('\n'.join(lines)))
audit = REPORTS['test']['audit']
shared_keys = ('mode', 'exploratory', 'platform_http_verified', 'packages', 'feature_count',
               'training_seconds', 'counts', 'independent_groups', 'identical_feature_vectors_shared')
shared = {key: audit.get(key) for key in shared_keys}
display(HTML('<details><summary>Shared configuration and data checks (expand)</summary><pre>'
             + html.escape(json.dumps(shared, ensure_ascii=False, indent=2)) + '</pre></details>'))
for split, report in REPORTS.items():
    interval = report.get('confidence_interval', {})
    if not interval.get('available'):
        print(split + ': ' + interval.get('reason', 'No confidence interval available.'))
print('All detailed metrics and audits remain in report.json/report.md and CSV files.')


In [ ]:
# One figure per split: matrix with count + row percentage, ROC, PR.
import csv
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is unavailable; all numeric results remain in CSV/JSON files.')
else:
    for split in ['test', 'validation'] + (['challenge'] if (OUTPUT / 'challenge/report.json').is_file() else []):
        folder = OUTPUT / split
        report = json.loads((folder / 'report.json').read_text(encoding='utf-8'))
        labels = ['normal', 'sqli', 'xss']
        matrix = report['confusion_matrix']
        fractions = [[v / sum(row) if sum(row) else 0 for v in row] for row in matrix]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
        ax = axes[0]
        image = ax.imshow(fractions, cmap='Blues', vmin=0, vmax=1)
        ax.set(xticks=range(3), yticks=range(3), xticklabels=labels, yticklabels=labels,
               xlabel='Predicted', ylabel='Actual', title='Count (row percentage)')
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f'{matrix[i][j]}\n({fractions[i][j]:.1%})', ha='center', va='center',
                        color='white' if fractions[i][j] > .5 else 'black')
        fig.colorbar(image, ax=ax, fraction=.046, pad=.04, label='Row proportion')
        with (folder / 'curves.csv').open(encoding='utf-8-sig', newline='') as handle:
            points = list(csv.DictReader(handle))
        for ax, curve in zip(axes[1:], ('ROC', 'PR')):
            for label in labels:
                values = [p for p in points if p['curve'] == curve and p['class'] == label]
                metric = report['one_vs_rest'][label].get('roc_auc' if curve == 'ROC' else 'average_precision')
                legend = f'{label} ({"AUC" if curve == "ROC" else "AP"}={metric:.3f})' if metric is not None else label
                xs, ys = [float(p['x']) for p in values], [float(p['y']) for p in values]
                if curve == 'PR':
                    ax.step(xs, ys, where='post', label=legend)
                else:
                    ax.plot(xs, ys, label=legend)
            if curve == 'ROC':
                ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
            ax.set(xlabel='FPR' if curve == 'ROC' else 'Recall',
                   ylabel='TPR' if curve == 'ROC' else 'Precision', title=curve,
                   xlim=(0, 1), ylim=(0, 1.02))
            ax.legend(fontsize=8)
        fig.suptitle(split + ' - ' + report['audit'].get('mode', 'unspecified'))
        fig.tight_layout()
        fig.savefig(folder / 'evaluation_overview.png', dpi=160)
        plt.show()
        plt.close(fig)


## 套件自我檢查與下載
`inference.zip` 內含模型、同版特徵／遮蔽／推論程式、依賴版本與自我檢查樣本。Windows、Ubuntu 各自解壓後都應先執行 `python predict.py --self-test`。Notebook 成功不等於已替你驗證所有作業系統。

In [ ]:
subprocess.check_call([sys.executable, str(OUTPUT / 'inference/predict.py'), '--self-test'])
import shutil
archive = shutil.make_archive(str(BASE / 'results_download'), 'zip', root_dir=OUTPUT)
files.download(archive)